# Augmented SBERT(AugSBERT)

## Overview
AugSBERT is  a hybrid approach that uses Cross-encoders to label large datasets and Bi-encoders to quickly and effectively handle inference, leading to a solution that maximizes both performance and scalability.

Augmented SBERT (AugSBERT) is designed to improve Bi-encoders by leveraging Cross-encoders to label or augment training datasets.

Depending on the amount of labeled data you have, there are three key scenarios in which AugSBERT can be applied—fully labeled datasets, limited labeled datasets, and completely unlabeled datasets. We will examine the implementation of the three key scenarios.


### 1) Full labeled sentence pairs dataset

Fine-tune a Sentence-BERT (SBERT) model using a combination of gold-standard sentence pairs and augmented data.

Our goal is to improve the model’s performance by leveraging synonym replacement as a data augmentation technique.

We'll use the STS benchmark dataset and the pre-trained BERT model as the backbone for sentence embedding generation.

### Importing Packages

In [1]:
import warnings
warnings.filterwarnings('ignore')

!pip install --upgrade pip -q
!pip uninstall -y transformers datasets tokenizers sentence-transformers nlpaug textattack flair -q
!pip install transformers sentence-transformers datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.1 MB/s eta 0:00:00


In [2]:
import os
import csv
import math
import gzip
import torch
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
#import nlpaug.augmenter.word as naw

from tqdm import tqdm
from datetime import datetime
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, models, losses, InputExample, SentencesDataset, util
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import Dataset

sns.set(style='darkgrid')

Note:

- sentence_transformers: Provides a high-level API to work with SBERT, including loading pre-trained models, pooling layers, and evaluation methods.
- nlpaug: A library for data augmentation, specifically ContextualWordEmbsAug used for synonym replacement with BERT.

### 1.2) Configuring the environment and model

We need to first set up the environment, choosing our pre-trained model, and configure the training parameters.

In [3]:
model_name = 'bert-base-uncased'
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 32
num_epochs = 5

we're choosing BERT-base-uncased as our pre-trained model, but this can be replaced with any transformer-based model. If you have access to the GPU, the code will run on it; otherwise, it will default to the CPU.

### 1.3) Preparing the data

Here, we load the STS Benchmark dataset, which contains human-annotated sentence pairs with similarity scores.


In [4]:
# Dataset paths and model save path
data_path = 'stsbenchmark.tsv.gz'
save_path = 'models/scenario1_model'

# Downloading the dataset
if not os.path.exists(data_path):
    util.http_get('https://sbert.net/datasets/stsbenchmark.tsv.gz', data_path)
    print("Dataset downloaded to {}".format(data_path))
else:
    print("Dataset already exists at {}".format(data_path))

Dataset downloaded to stsbenchmark.tsv.gz


We convert the dataset into a format that our model can work with. This involves reading the dataset and splitting it into two sets: training and validation.

In [5]:
# Converting the dataset into training and validation

train, validation = [],[]
with gzip.open(data_path, 'rt', encoding='utf8') as fIn:
    reader = csv.DictReader(fIn, delimiter='\t', quoting=csv.QUOTE_NONE)
    for row in reader:
      x = [row['sentence1'], row['sentence2']]
      y = float(row['score']) / 5.0

      sample = InputExample(texts=x, label=y)

      if row['split'] == 'dev':
        validation.append(sample)
      elif row['split']== 'train':
        train.append(sample)

print("Train examples: {}".format(len(train)))
print("Validation examples: {}".format(len(validation)))
print(train[0])


Train examples: 5749
Validation examples: 1500
<InputExample> label: 1.0, texts: A plane is taking off.; An air plane is taking off.


### 1.4)  Data Augmentation

To improve our model’s generalization, we use synonym replacement as a data augmentation strategy.

For this, we use BERT to generate contextually appropriate synonyms and insert them into our sentence pairs, creating augmented data.

In [6]:
# The nlpaug library is incompatible with the current transformers version.
# Disabling nlpaug code and will implement custom augmentation using transformers.
# aug = naw.ContextualWordEmbsAug(model_path=model_name, action="insert", device=device)

# augmented = []

# # Augment training samples with synonym replacement
# for sample in tqdm(train, unit="docs"):
#   augmented_texts = aug.augment(sample.texts)
#   inp_example = InputExample(texts=augmented_texts, label=sample.label)
#   augmented.append(inp_example)

# print("Augmented training examples: {}".format(len(augmented)))

### 1.4) Custom Data Augmentation with Masked Language Model (MLM)

 This function will use a pre-trained Masked Language Model (MLM) from the `transformers` library to insert new words into sentences based on context. This provides a similar effect to `nlpaug.ContextualWordEmbsAug`.

In [7]:
from transformers import BertTokenizer, BertForMaskedLM
import torch
import random

# Load pre-trained tokenizer and model for MLM
tokenizer = BertTokenizer.from_pretrained(model_name)
mlm_model = BertForMaskedLM.from_pretrained(model_name).to(device)
mlm_model.eval()

def custom_mlm_augment(sentence, num_insertions=1, top_k=5):
    tokenized_sentence = tokenizer.encode(sentence, add_special_tokens=True, return_tensors='pt')
    input_ids = tokenized_sentence[0].tolist()

    augmented_sentence = list(input_ids)

    for _ in range(num_insertions):
        # Choose a random position to insert a mask token (excluding special tokens)
        insertion_idx = random.randint(1, len(augmented_sentence) - 2)

        # Insert MASK token
        masked_input = augmented_sentence[:insertion_idx] + [tokenizer.mask_token_id] + augmented_sentence[insertion_idx:]

        # Convert to tensor and move to device
        input_ids_tensor = torch.tensor([masked_input]).to(device)

        with torch.no_grad():
            outputs = mlm_model(input_ids_tensor)
            predictions = outputs.logits

        # Get probabilities for the masked position
        masked_token_logits = predictions[0, insertion_idx].cpu().numpy()

        # Get top_k predicted tokens
        top_k_tokens = np.argsort(-masked_token_logits)[:top_k]

        # Choose one of the top_k tokens randomly to insert
        chosen_token = random.choice(top_k_tokens)

        # Replace MASK token with the chosen token
        augmented_sentence[insertion_idx] = chosen_token

    # Decode the augmented sentence back to a string
    return tokenizer.decode(augmented_sentence, skip_special_tokens=True)

augmented = []

# Augment training samples
print("Augmenting training examples...")
for sample in tqdm(train, unit="docs"):
    original_sentence1 = sample.texts[0]
    original_sentence2 = sample.texts[1]

    # Augment sentence1
    augmented_sentence1 = custom_mlm_augment(original_sentence1, num_insertions=1, top_k=5)

    # Create a new InputExample with augmented sentence1 and original sentence2
    inp_example = InputExample(texts=[augmented_sentence1, original_sentence2], label=sample.label)
    augmented.append(inp_example)

print("Original training examples: {}".format(len(train)))
print("Augmented training examples with Custom MLM: {}".format(len(augmented)))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Augmenting training examples...


100%|██████████| 5749/5749 [15:45<00:00,  6.08docs/s]

Original training examples: 5749
Augmented training examples with Custom MLM: 5749


In the code above, we use the ContextualWordEmbsAug method to augment our original training samples by replacing words with synonyms generated by BERT.

This increases the training data size, which helps the Bi-encoder learn better representations.

### 1.5) Create and Configure SBERT Model

Now, we set up our SBERT model. As discussed earlier, SBERT uses a pre-trained transformer (like BERT) to map tokens to embeddings, and we apply mean pooling to obtain fixed-size sentence embeddings.

In [8]:
# Load pre-trained BERT model for token embeddings
word_embedding_model = models.Transformer(model_name)

# Apply mean pooling to get one fixed-sized sentence vector
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(),
                               pooling_mode_mean_tokens=True,
                               pooling_mode_cls_token=False,
                               pooling_mode_max_tokens=False)

# Combining the transformer and pooling into a SentenceTransformer model
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Above, we created the Bi-encoder using the SentenceTransformer framework, which combines the transformer model and the pooling mechanism. The model maps each sentence to a fixed-size vector.

### 1.6) Prepare the DataLoader and Training configuration

We need to prepare the dataset for training. The entire data (train + augmented) is combined, and a DataLoader is created to feed it into the model.

In [9]:
# Combine train and augmented samples
train_dataset = SentencesDataset(train + augmented, model)
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size = batch_size)

# Use cosine similarity loss for training
train_loss = losses.CosineSimilarityLoss(model=model)

# Set up evaluator to monitor progress on the validation set
evaluator = EmbeddingSimilarityEvaluator.from_input_examples(validation, name='sts-dev')


We define the loss function as CosineSimilarityLoss, which ensures that the embeddings for similar sentence pairs are close to each other in vector space.

### 1.7) Train the Model

We configure the training procedure by specifying the epochs and evaluation frequency, and then we start the training process as follows:

In [ ]:
# Train the model
model.fit(train_objectives=[(train_dataloader, train_loss)],
          evaluator=evaluator,
          epochs=num_epochs,
          evaluation_steps=1000,
          output_path=save_path)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss


Only train on the training data (as follows):

In [ ]:
# Combine train samples
train_dataset = SentencesDataset(train, model)
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)

# Use cosine similarity loss for training
train_loss = losses.CosineSimilarityLoss(model=model)

# Set up evaluator to monitor progress on the validation set
evaluator = EmbeddingSimilarityEvaluator.from_input_examples(validation, name='sts-dev')

# Train the model
model.fit(train_objectives=[(train_dataloader, train_loss)],
          evaluator=evaluator,
          epochs=num_epochs,
          evaluation_steps=1000,
          output_path=save_path)

If you look closely, every metric with training set is lower than the corresponding metric for train + augmented set (look at the last row in both tables), which means that train+augmented leads to better performance than training data alone.

# 2) Few annotated samples

In this case, we have limited labeled dataset.

- Firstly we train a Cross-encoder on the small labeled dataset,

 - Use it to label additional data (creating "silver" samples),

 - Train a Bi-encoder on the combined labeled and silver datasets.

### 2.1) Imports and configuration

In [ ]:

from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CECorrelationEvaluator
import math

# Configuration
model_name = 'bert-base-uncased'
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 32
num_epochs = 5

data_path = 'datasets/stsbenchmark.tsv.gz'
cross_encoder_path = 'models/scenario2_model_cross_encoder'
bi_encoder_path = 'models/scenario2_model_bi_encoder'

### 2.2) Preparing the data

We’ll load and split the STS Benchmark dataset into training (train) and validation (validation) sets. The Cross-encoder will be trained on the training dataset, and the validation set will be used for evaluation.

In [ ]:
train, validation = [], []

# Read the STS benchmark dataset
with gzip.open(data_path, 'rt', encoding='utf8') as f:
    reader = csv.DictReader(f, delimiter='\t', quoting=csv.QUOTE_NONE)

    for row in reader:
        y = float(row['score']) / 5.0
        example1 = InputExample(texts=[row['sentence1'], row['sentence2']], label=y)
        example2 = InputExample(texts=[row['sentence2'], row['sentence1']], label=y)

        if row['split'] == 'dev':
            validation.append(example1)

        elif row['split'] == 'train':
            # Ensure symmetric pairs, i.e. CrossEncoder(A,B) = CrossEncoder(B,A)
            train.append(example1)
            train.append(example2)


This ensures that each pair in the training dataset is symmetric, as Cross-encoders should generate the same similarity score for CrossEncoder(A,B) and CrossEncoder(B,A).

### 2.3) Train the Cross-encoder

we train a Cross-encoder on the small training dataset. This model will later be used to weakly label additional sentence pairs (silver samples) for Bi-encoder training.

In [ ]:
# Initialize the Cross-encoder model
cross_encoder = CrossEncoder(model_name, num_labels=1)

# Create a DataLoader for the training data
train_dataloader = DataLoader(train, shuffle=True, batch_size=batch_size)

# Set up an evaluator to monitor progress on the validation set
evaluator = CECorrelationEvaluator.from_input_examples(validation, name='sts-dev')

# Train the Cross-encoder model
cross_encoder.fit(train_dataloader=train_dataloader,
          evaluator=evaluator,
          epochs=num_epochs,
          evaluation_steps=1000,
          output_path=cross_encoder_path)

Here, the Cross-encoder is trained using the STS Benchmark training set. We also use a validation set evaluator to measure how well the model performs on the validation set.

### 2.4) Prepare silver pairs for labeling

With the Cross-encoder trained, we now need to generate additional sentence pairs to label with it. These pairs will act as "silver" data. We’ll use a pre-trained Bi-encoder to retrieve the top-k most similar sentences for each sentence in the training set.

In [ ]:
# Initialize variables for silver pair creation
augmented = []
sentences = set()

# Number of top-k similar sentences to retrieve
top_k = 3


- augmented: This list will store the new sentence pairs that we generate.

- sentences: This set will be used to collect all the unique sentences from the training dataset.

- top_k: This specifies how many similar sentences we want to retrieve for each sentence. In this case, we’ll retrieve the top 3 most similar sentences for each sentence in our dataset.

In [ ]:
for sample in train:
    sentences.update(sample.texts)

# Convert to list and create sentence-to-index mapping
sentences = list(sentences)
sent2idx = {sentence: idx for idx, sentence in enumerate(sentences)}


We also need to ensure that we don’t accidentally add any of the sentence pairs that already exist in the original training set. For this, we create a set called duplicates:

In [ ]:
duplicates = set((sent2idx[data.texts[0]], sent2idx[data.texts[1]]) for data in train)


The duplicates set contains the indexes of the sentence pairs that already exist in the training data. We use this to prevent adding these pairs again to the silver data.


To find similar sentences, we use a pre-trained Bi-encoder model. In this case, we’ll use bert-base-nli-stsb-mean-tokens, which is a Sentence-BERT model pre-trained on the STS benchmark and other similar tasks.

In [ ]:
# Use a pre-trained model for semantic search
semantic_search_model = SentenceTransformer('bert-base-nli-stsb-mean-tokens')

This model generates sentence embeddings, which we will use to calculate the similarity between sentences in our dataset.

We now pass all the unique sentences through the Bi-encoder to generate their corresponding embeddings. These embeddings represent the sentences as fixed-size vectors in a high-dimensional space.

In [ ]:
# Encode all unique sentences
embeddings = semantic_search_model.encode(sentences,
                                          batch_size=batch_size,
                                          convert_to_tensor=True)

The encode method transforms each sentence into a dense vector that captures its semantic meaning. These embeddings will allow us to measure the similarity between any two sentences using cosine similarity.


For each sentence in our dataset, we calculate its cosine similarity with every other sentence in the dataset. We then retrieve the top-k most similar sentences using torch.topk:

In [ ]:
# Perform top-k retrieval of similar sentences
for idx in tqdm(range(len(sentences)), unit="docs"):
    sentence_embedding = embeddings[idx]
    cos_scores = util.pytorch_cos_sim(sentence_embedding, embeddings)[0]
    cos_scores = cos_scores.cpu()

    # Retrieve the top-k most similar sentences (excluding the sentence itself)
    top_results = torch.topk(cos_scores, k=top_k+1)

    for score, iid in zip(top_results[0], top_results[1]):
        if iid != idx and (iid, idx) not in duplicates:
            augmented.append((sentences[idx], sentences[iid]))
            duplicates.add((idx, iid))

We should note that:

- sentence_embedding = embeddings[idx]: For each sentence, we retrieve its embedding.

- cos_scores = util.pytorch_cos_sim(sentence_embedding, embeddings)[0]: We compute the cosine similarity between the current sentence and every other sentence in the dataset. The result is a list of similarity scores.

- top_results = torch.topk(cos_scores, k=top_k+1): We use torch.topk to retrieve the top-k most similar sentences for the current sentence. We add 1 to the k value because the most similar sentence will always be the sentence itself (which we will later exclude).

- Avoiding Duplicates: For each retrieved sentence pair, we check whether it is a duplicate (i.e., already present in the original training set) using the duplicates set. If the pair isn’t already present, we add it to the augmented list and mark it as added to the duplicates set.


After running this process, the augmented list will contain new sentence pairs that are similar but were not part of the original training set.

These sentence pairs are ready to be labeled by the Cross-encoder, generating the silver data that will be used to train the Bi-encoder in the next steps.

### 2.5) Label silver pairs using the cross-encoder

With the new sentence pairs generated, we use the trained Cross-encoder to label these "silver" pairs

In [ ]:
cross_encoder = CrossEncoder(cross_encoder_path)
silver_scores = cross_encoder.predict(augmented)


The Cross-encoder assigns similarity scores to the newly created sentence pairs, effectively weakly labeling the "silver" data.

### 2.6) Train the Bi-encoder

To train the bi-encoder, first, after generating the silver sentence pairs in the previous step and obtaining the similarity scores from the Cross-encoder, we need to package these pairs into the format expected by the InputExample class.

In [ ]:
# Prepare silver samples in the required format
aug_samples = [InputExample(texts=[pair[0], pair[1]], label=score) \
               for pair, score in zip(augmented, silver_scores)]

- aug_samples: This list contains the new silver pairs with their corresponding similarity scores.

- pair[0], pair[1]: These are the sentence pairs we retrieved through semantic search.

- score: This is the similarity score predicted by the Cross-encoder for each pair.


Next, we define the Bi-encoder, which consists of a BERT-based transformer to encode the sentences and a mean pooling layer to generate fixed-size sentence embeddings.

In [ ]:
# define the bi-encoder model
word_embedding_model = models.Transformer(model_name, max_seq_length=512)

pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(),
                               pooling_mode_mean_tokens=True)

bi_encoder = SentenceTransformer(modules=[word_embedding_model, pooling_model])

- word_embedding_model: This is the pre-trained transformer model (in this case, BERT) that will be used to map each sentence to a high-dimensional embedding space.

- pooling_model: Since the transformer produces embeddings for each token in the sentence, we apply mean pooling to aggregate these token embeddings into a single vector that represents the entire sentence.

- SentenceTransformer: This is the framework that combines the transformer and pooling layers into a single Bi-encoder model.


Now that we have both the gold-labeled dataset and the silver-labeled samples, we combine them into one unified dataset that will be used to train the Bi-encoder.

In [ ]:
# Combine gold and silver samples
train_dataset = SentencesDataset(train + aug_samples, bi_encoder)
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)

- train + aug_samples: We concatenate the original gold-labeled training data with the silver samples generated by the Cross-encoder.

- SentencesDataset: This class prepares the combined dataset in a format compatible with the SentenceTransformer framework.

- DataLoader: We wrap the combined dataset in a DataLoader to shuffle and batch the data during training. This ensures that the model gets diverse samples during each training step and helps prevent overfitting.



Like we did in scenario 1, for training, we use cosine similarity loss, which ensures that the model produces embeddings for similar sentence pairs that are close together in the embedding space, and embeddings for dissimilar pairs that are far apart.

In [ ]:
# Set loss function and evaluator for Bi-encoder
train_loss = losses.CosineSimilarityLoss(model=bi_encoder)
evaluator = EmbeddingSimilarityEvaluator.from_input_examples(validation, name='sts-dev')

#### Finally, we configure and run the training loop for the Bi-encoder.

In [ ]:
# Train the Bi-encoder
bi_encoder.fit(train_objectives=[(train_dataloader, train_loss)],
               evaluator=evaluator,
               epochs=num_epochs,
               evaluation_steps=1000,
               output_path=bi_encoder_path)

Note:

The entire code until section 2.5 remains the same, the only thing we change is the following piece of code, wherein we only use the train set:

In [ ]:
# Prepare silver samples in the required format
aug_samples = [InputExample(texts=[pair[0], pair[1]], label=score) \
               for pair, score in zip(augmented, silver_scores)]

# define the bi-encoder model
word_embedding_model = models.Transformer(model_name, max_seq_length=512)

pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(),
                               pooling_mode_mean_tokens=True)

bi_encoder = SentenceTransformer(modules=[word_embedding_model, pooling_model])

# Combine gold and silver samples
train_dataset = SentencesDataset(train, bi_encoder)
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)

# Set loss function and evaluator for Bi-encoder
train_loss = losses.CosineSimilarityLoss(model=bi_encoder)
evaluator = EmbeddingSimilarityEvaluator.from_input_examples(validation, name='sts-dev')

# Train the Bi-encoder
bi_encoder.fit(train_objectives=[(train_dataloader, train_loss)],
               evaluator=evaluator,
               epochs=num_epochs,
               evaluation_steps=1000,
               output_path=bi_encoder_path)

Every metric with just the training set is lower than the corresponding metric for train + augmented set (look at the last row in both tables), which means that train+augmented leads to better performance than training data alone.

This shows that augmenting the original dataset with silver data allows the Bi-encoder to learn from a more diverse set of sentence pairs, since it leverages the strengths of both Cross-encoders and Bi-encoders.

## 3) No annotated samples